In [1]:
import pandas as pd 
import numpy as np 
import os 

In [2]:
df_prods = pd.read_csv(r'/Users/mariatirado/Downloads/07-2024 Instacart Basket Analysis/02 Data/Original Data/4/products.csv', index_col=False)

In [3]:
df_ords = pd.read_csv(r'/Users/mariatirado/29–10–2025 Instacart Basket Analysis/02 Data/Prepared Data/orders_wrangled.csv', index_col=False)

# 01 Test 

In [4]:
df_test = pd.DataFrame()

In [5]:
df_test['mix'] = ['a', 'b', 1, True]

In [6]:
df_test.head()

,mix
0,a
1,b
2,1
3,True


In [7]:
for col in df_test.columns.tolist():
  weird = (df_test[[col]].map(type) != df_test[[col]].iloc[0].map(type)).any(axis=1)
  if len (df_test[weird]) > 0:
    print (col)

mix


# checking whether the dataframe contains any mixed-type columns

In [8]:
df_test['mix'] = df_test['mix'].astype('str')

# Converting column data type

In [9]:
df_prods.isnull().sum()

product_id        0
product_name     16
aisle_id          0
department_id     0
prices            0
dtype: int64

# Creating new data frame

In [10]:
df_nan = df_prods[df_prods['product_name'].isnull() == True]

In [11]:
df_nan

,product_id,product_name,aisle_id,department_id,prices
33,34,NaN,121,14,12.2
68,69,NaN,26,7,11.8
115,116,NaN,93,3,10.8
261,262,NaN,110,13,12.1
525,525,NaN,109,11,1.2
1511,1511,NaN,84,16,14.3
1780,1780,NaN,126,11,12.3
2240,2240,NaN,52,1,14.2
2586,2586,NaN,104,13,12.4
3159,3159,NaN,126,11,13.1


In [12]:
df_prods.shape

(49693, 5)

In [13]:
df_prods_clean = df_prods[df_prods['product_name'].isnull() == False]

In [14]:
df_prods_clean.shape

(49677, 5)

In [15]:
df_dups = df_prods_clean[df_prods_clean.duplicated()]

In [16]:
df_dups

,product_id,product_name,aisle_id,department_id,prices
462,462,Fiber 4g Gummy Dietary Supplement,70,11,4.8
18459,18458,Ranger IPA,27,5,9.2
26810,26808,Black House Coffee Roasty Stout Beer,27,5,13.4
35309,35306,Gluten Free Organic Peanut Butter & Chocolate ...,121,14,6.8
35495,35491,Adore Forever Body Wash,127,11,9.9


In [17]:
df_prods_clean.shape

(49677, 5)

# Addressing duplicates 

In [18]:
df_prods_clean_no_dups = df_prods_clean.drop_duplicates()

In [19]:
df_prods_clean_no_dups.shape

(49672, 5)

# 01 Run the df.describe() function on your df_ords dataframe

In [20]:
df_ords.describe()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01


# Answer
After running the df.describe() function, most columns in the df_ords dataframe appear consistent 
However, the order_number column shows a maximum value of 100, which might be unusually high. This could indicate that some users have placed an unrealistic number of orders and should be investigated further.

# 02 Check for mixed-type data in your df_ords dataframe.

In [21]:
for col in df_ords.columns.tolist():
    weird = df_ords[col].map(type) != type(df_ords[col].iloc[0])
    if weird.any():
        print(col)

order_id
user_id
order_number
order_dow
order_hour_of_day
days_since_prior_order


# 03 If you find mixed-type data, fix it

In [22]:
expected = {
    'order_id': 'Int64',           # nullable integer
    'user_id': 'Int64',
    'order_number': 'Int64',
    'order_dow': 'Int64',          # 0..6
    'order_hour_of_day': 'Int64',  # 0..23
    'days_since_prior_order': 'Float64'  # allows NaN
}

In [23]:
for col in df_ords.columns.tolist():
    weird = df_ords[col].map(type) != type(df_ords[col].iloc[0])
    if weird.any():
        print(col)

order_id
user_id
order_number
order_dow
order_hour_of_day
days_since_prior_order


In [24]:
import numpy as np
import pandas as pd

targets = {
    'order_id': 'Int64',
    'user_id': 'Int64',
    'order_number': 'Int64',
    'order_dow': 'Int64',
    'order_hour_of_day': 'Int64',
    'days_since_prior_order': 'Float64'}

for col, tgt in targets.items():
    # 1) normalize texty placeholders and strip whitespace
    s = df_ords[col].astype(str).str.strip()
    s = s.replace({'': np.nan, 'NA': np.nan, 'NaN': np.nan, 'None': np.nan, 'missing': np.nan})
    # 2) remove visual junk (commas, currency, spaces)
    s = s.str.replace(r'[,\s$€]', '', regex=True)
    # 3) convert to numeric, coerce bad values to NaN, then cast to target dtype
    s = pd.to_numeric(s, errors='coerce')
    df_ords[col] = s.astype(tgt)

df_ords.dtypes 

order_id                    Int64
user_id                     Int64
eval_set                   object
order_number                Int64
order_dow                   Int64
order_hour_of_day           Int64
days_since_prior_order    Float64
dtype: object

In [25]:
for col in df_ords.columns:
    non_na = df_ords[col].dropna()
    if non_na.empty:
        continue
    ref = type(non_na.iloc[0])
    if (df_ords[col].map(type) != ref).any():
        print("Still mixed:", col, df_ords[col].map(type).unique())


Still mixed: order_id [<class 'int'>]
Still mixed: user_id [<class 'int'>]
Still mixed: order_number [<class 'int'>]
Still mixed: order_dow [<class 'int'>]
Still mixed: order_hour_of_day [<class 'int'>]
Still mixed: days_since_prior_order [<class 'float'>]


# Replace values if the data is not clean - Code saved for future tasks 

# 04 Run a check for missing values in your df_ords dataframe.

In [26]:
df_ords.isnull().sum()

order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

# Answer 
For a customer’s first order, there is no “previous order,” so it makes sense that this field is empty for those rows.

# 05 Address the missing values using an appropriate method.

In [27]:
df_ords[df_ords['days_since_prior_order'].isnull()]


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,<NA>
11,2168274,2,prior,1,2,11,<NA>
26,1374495,3,prior,1,1,14,<NA>
39,3343014,4,prior,1,6,11,<NA>
45,2717275,5,prior,1,3,12,<NA>
...,...,...,...,...,...,...,...
3420930,969311,206205,prior,1,4,12,<NA>
3420934,3189322,206206,prior,1,3,18,<NA>
3421002,2166133,206207,prior,1,6,19,<NA>
3421019,2227043,206208,prior,1,1,15,<NA>


In [28]:
df_ords.loc[df_ords['days_since_prior_order'].isna(), 'days_since_prior_order'] = 0

In [29]:
df_ords[df_ords['days_since_prior_order'].isnull()]

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order


# Answer
Since these customers had no prior order, the missing values logically represent zero days since a previous order.

# 06 Run a check for duplicate values in your df_ords data.

In [30]:
df_dups = df_ords[df_ords.duplicated()]

In [31]:
df_dups

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order


In [32]:
df_ords.duplicated().sum()

np.int64(0)

# Answer
There are no duplicate rows in the dataset.
This suggests that each order record in df_ords is unique and there are no data-entry errors caused by duplication.

In [34]:
from pathlib import Path

home = Path("/Users/mariatirado")

# Show the two similarly named project folders, if both exist
candidates = list(home.glob("29*Instacart Basket Analysis"))
print("Project folders I can see:")
for c in candidates:
    print(" •", c, " | has Prepared Data:", (c / "02 Data" / "Prepared Data").exists())

# Find the CSVs anywhere under your home
orders_hits = list(home.glob("**/02 Data/Prepared Data/orders_wrangled.csv"))
prods_hits  = list(home.glob("**/02 Data/Prepared Data/products_cleaned.csv"))

print("\norders_wrangled.csv found at:")
for p in orders_hits: print(" •", p)

print("\nproducts_cleaned.csv found at:")
for p in prods_hits: print(" •", p)


Project folders I can see:
 • /Users/mariatirado/29–10–2025 Instacart Basket Analysis  | has Prepared Data: True
 • /Users/mariatirado/29-10-2025 Instacart Basket Analysis  | has Prepared Data: True

orders_wrangled.csv found at:
 • /Users/mariatirado/29–10–2025 Instacart Basket Analysis/02 Data/Prepared Data/orders_wrangled.csv

products_cleaned.csv found at:
 • /Users/mariatirado/29–10–2025 Instacart Basket Analysis/02 Data/Prepared Data/products_cleaned.csv


In [35]:
import os
import pandas as pd

# ✅ Correct path (with en dashes)
path = r'/Users/mariatirado/29–10–2025 Instacart Basket Analysis'

# Load the existing CSVs
df_ords = pd.read_csv(os.path.join(path, '02 Data', 'Prepared Data', 'orders_wrangled.csv'))
df_prods = pd.read_csv(os.path.join(path, '02 Data', 'Prepared Data', 'products_cleaned.csv'))

print("Orders shape:", df_ords.shape)
print("Products shape:", df_prods.shape)


Orders shape: (3421083, 7)
Products shape: (49693, 5)


In [36]:
# Ensure folder exists
os.makedirs(os.path.join(path, '02 Data', 'Prepared Data'), exist_ok=True)

# Export cleaned versions
df_ords.to_csv(os.path.join(path, '02 Data', 'Prepared Data', 'orders_cleaned.csv'), index=False)
df_prods.to_csv(os.path.join(path, '02 Data', 'Prepared Data', 'products_cleaned.csv'), index=False)

print("✅ Files exported successfully to the 'Prepared Data' folder!")


✅ Files exported successfully to the 'Prepared Data' folder!
